In [1]:
# FINAL PIPELINE

In [2]:
# Installing the Dependency -

!pip install groq # for API
!pip install pdfplumber # PDF Data Extraction
!pip install json-repair # JSON VALIDATION
!pip install pydantic # For Resume Evaluation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 2.2 MB/s eta 0:00:00


In [3]:
# Importing Nessaceery Libraries

import json
import pdfplumber

from groq import Groq
from google.colab import files
from json_repair import repair_json
from pydantic import BaseModel
from typing import List

In [5]:
#  Basically Here i am Configuring the Groc API for Gobal Access

client = Groq(
    api_key="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"
)

In [7]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": "Say hello"
        }
    ]
)

print(response.choices[0].message.content)

# Just checking wether our API is working or not!

Hello. How can I help you today?


In [9]:
#  Create Basic Structure for Resume Result Evauation

class FinalEvaluation(BaseModel):

    match_score: int
    matched_skills: List[str]
    missing_skills: List[str]
    summary: str
    recommendation: str

In [10]:
# Asking the Resume from User
print("Upload Resume PDF")

uploaded = files.upload()

uploaded_file_name = list(uploaded.keys())[0]

Upload Resume PDF


Saving Vraj_Suratwala_Resume_MSCIT.pdf to Vraj_Suratwala_Resume_MSCIT.pdf


In [11]:
def extract_resume_text(pdf_file):

    resume_text = ""

    with pdfplumber.open(pdf_file) as pdf:

        for page in pdf.pages:

            text = page.extract_text()

            if text:
                resume_text += text + "\n"

    return resume_text

In [12]:
try:

    resume_text = extract_resume_text(uploaded_file_name)

    if len(resume_text.strip()) < 50:
        raise Exception("Resume text too short or unreadable")

    print("\nResume Text Extracted Successfully")
    print(f"Extracted {len(resume_text)} characters")

except Exception as e:

    print("\nPDF Extraction Failed")
    print(str(e))


Resume Text Extracted Successfully
Extracted 5625 characters


In [13]:
job_description = input("\nEnter Job Description:\n")


Enter Job Description:
We are hiring a Full Stack Python Developer.  Required Skills: - Python - Flask - SQL - REST APIs - HTML - CSS - JavaScript  Preferred Skills: - Docker - AWS - React - Git  Experience Required: 1-3 years  Responsibilities: - Develop backend APIs using Flask - Design and manage databases - Build responsive frontend interfaces - Integrate APIs with frontend systems - Debug and optimize web applications - Collaborate with development teams


In [14]:
def analyze_job_description(job_description):

    jd_prompt = f"""
    Analyze the following job description.

    Return ONLY valid JSON.

    JSON Schema:
    {{
        "role": "",
        "required_skills": [],
        "preferred_skills": [],
        "experience_required": ""
    }}

    Job Description:
    {job_description}
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": jd_prompt
            }
        ]
    )

    raw_output = response.choices[0].message.content

    fixed_json = repair_json(raw_output)

    return json.loads(fixed_json)

In [15]:
def analyze_resume_details(resume_text):

    resume_prompt = f"""
    Analyze the following resume carefully.

    Return ONLY valid JSON.

    JSON Schema:
    {{
        "candidate_name": "",
        "skills": [],
        "projects": [],
        "experience": "",
        "education": ""
    }}

    Rules:
    - Extract technical skills only
    - Keep project names short
    - Do not return markdown
    - Do not return explanation text

    Resume:
    {resume_text}
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": resume_prompt
            }
        ]
    )

    raw_output = response.choices[0].message.content

    fixed_json = repair_json(raw_output)

    return json.loads(fixed_json)

In [16]:
def match_resume_to_jd(jd_analysis, resume_analysis):

    matching_prompt = f"""
    Compare the candidate resume with the job description.

    Return ONLY valid JSON.

    JSON Schema:
    {{
        "match_score": 0,
        "matched_skills": [],
        "missing_skills": [],
        "summary": "",
        "recommendation": ""
    }}

    Rules:
    - match_score must be between 0 and 100
    - recommendation must be one of:
        - Strong Fit
        - Potential Fit
        - Not a Fit
    - Consider related technologies as partial matches
    - Example:
        - SQLite counts as partial SQL experience
        - FastAPI counts similar to Flask
    - Summary should be 2-3 lines
    - Do not return markdown
    - Do not return explanation text

    Job Description Analysis:
    {json.dumps(jd_analysis, indent=4)}

    Resume Analysis:
    {json.dumps(resume_analysis, indent=4)}
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": matching_prompt
            }
        ]
    )

    raw_output = response.choices[0].message.content

    fixed_json = repair_json(raw_output)

    return json.loads(fixed_json)

In [17]:
def validate_final_output(final_result):

    validated_result = FinalEvaluation(**final_result)

    return validated_result

In [18]:
def apply_recommendation_logic(validated_result):

    score = validated_result.match_score

    if score >= 80:
        validated_result.recommendation = "Strong Fit"

    elif score >= 60:
        validated_result.recommendation = "Potential Fit"

    else:
        validated_result.recommendation = "Not a Fit"

    return validated_result

In [19]:
def save_report(validated_result):

    with open("resume_evaluation.json", "w") as file:

        json.dump(
            validated_result.model_dump(),
            file,
            indent=4
        )

    print("\nJSON Report Saved Successfully")

In [31]:
# =====================================
# FINAL PIPELINE EXECUTION
# =====================================

try:

    print("\n=====================================")
    print(" AI Resume Screening Agent Started ")
    print("=====================================\n")


    # =====================================
    # STEP 1 — Analyze Job Description
    # =====================================

    print("Analyzing Job Description...\n")

    jd_analysis = analyze_job_description(
        job_description
    )

    print("JD Analysis Completed Successfully\n")


    # =====================================
    # STEP 2 — Analyze Resume
    # =====================================

    print("Analyzing Resume...\n")

    resume_analysis = analyze_resume_details(
        resume_text
    )

    print("Resume Analysis Completed Successfully\n")


    # =====================================
    # STEP 3 — Match Resume with JD
    # =====================================

    print("Matching Resume with Job Description...\n")

    final_result = match_resume_to_jd(
        jd_analysis,
        resume_analysis
    )

    print("Resume Matching Completed Successfully\n")


    # =====================================
    # STEP 4 — Validate Final Output
    # =====================================

    print("Validating AI Response...\n")

    validated_result = validate_final_output(
        final_result
    )

    print("Validation Completed Successfully\n")


    # =====================================
    # STEP 5 — Apply Recommendation Logic
    # =====================================

    print("Applying Recommendation Logic...\n")

    validated_result = apply_recommendation_logic(
        validated_result
    )

    print("Recommendation Logic Applied Successfully\n")


    # =====================================
    # STEP 6 — Display Final Result
    # =====================================

    print("=====================================")
    print(" FINAL RESUME EVALUATION RESULT ")
    print("=====================================\n")

    print(
        validated_result.model_dump_json(
            indent=4
        )
    )

    print(
        f"\nFinal Match Score: {validated_result.match_score}/100"
    )


    # =====================================
    # STEP 7 — Career Suggestions
    # =====================================

    if validated_result.match_score < 70:

        print("\n=====================================")
        print(" CAREER SUGGESTIONS ")
        print("=====================================\n")

        print("Suggested Platforms:")
        print("1. LinkedIn Jobs  → https://www.linkedin.com/jobs/")
        print("2. Unstop         → https://unstop.com/jobs")
        print("3. Internshala    → https://internshala.com/")


    # =====================================
    # STEP 8 — Save JSON Report
    # =====================================

    print("\nSaving JSON Evaluation Report...\n")

    save_report(validated_result)

    print("\nReport Saved Successfully")


    # =====================================
    # PIPELINE COMPLETED
    # =====================================

    print("\n=====================================")
    print(" AI Resume Screening Completed ")
    print("=====================================\n")


except Exception as e:

    print("\n=====================================")
    print(" PIPELINE FAILED ")
    print("=====================================\n")

    print("Error:")
    print(str(e))


 AI Resume Screening Agent Started 

Analyzing Job Description...

JD Analysis Completed Successfully

Analyzing Resume...

Resume Analysis Completed Successfully

Matching Resume with Job Description...

Resume Matching Completed Successfully

Validating AI Response...

Validation Completed Successfully

Applying Recommendation Logic...

Recommendation Logic Applied Successfully

 FINAL RESUME EVALUATION RESULT 

{
    "match_score": 60,
    "matched_skills": [
        "Python",
        "SQL",
        "JavaScript"
    ],
    "missing_skills": [
        "Flask",
        "REST APIs",
        "HTML",
        "CSS"
    ],
    "summary": "Candidate has experience with Python, SQL, and JavaScript, but lacks experience with Flask, REST APIs, HTML, and CSS.",
    "recommendation": "Potential Fit"
}

Final Match Score: 60/100

 CAREER SUGGESTIONS 

Suggested Platforms:
1. LinkedIn Jobs  → https://www.linkedin.com/jobs/
2. Unstop         → https://unstop.com/jobs
3. Internshala    → https://in

In [ ]:
#  Thank you!